# Data 140: Intro to Machine Learning (Interactive)

## How to use this notebook

**Important:** Students should interact with **widgets** (checkboxes/sliders) instead of editing code.

### Step-by-step workflow
1. Run cells **top to bottom** once.
2. In each interactive section, change widget values.
3. Record results in the reflection prompts.
4. Do **not** modify code unless your instructor asks you to.

---

## Learning goals
- Explain the difference between **features (X)** and **label (y)**.
- Observe how feature choice changes model quality.
- Compare regression and classification workflows.


In [ ]:
# Setup (run once)
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
import pandas as pd

from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
except ImportError as e:
    raise ImportError(
        "This notebook requires ipywidgets. Install it with: pip install ipywidgets"
    ) from e


# Part 1) Supervised Learning — Regression

We will predict California median house value (`MedHouseVal`).

## Dataset features
- `MedInc`: median income
- `HouseAge`: median house age
- `AveRooms`: average rooms
- `AveBedrms`: average bedrooms
- `Population`: neighborhood population
- `AveOccup`: average occupancy
- `Latitude`, `Longitude`: location coordinates

### Your task
Use the widgets to test different feature combinations and test sizes.


In [ ]:
# Load and preview California housing data
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

display(df.head())
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")


## Interactive Regression Lab

### Exactly what to do
1. In **Features to include**, select 1+ features.
2. Move the **Test set size** slider.
3. Read the model output below.
4. Repeat at least 4 trials.

### What to look for
- Lower **MSE** is better.
- Compare whether adding more features helps every time.


In [ ]:
all_regression_features = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"
]

reg_feature_widget = widgets.SelectMultiple(
    options=all_regression_features,
    value=("MedInc", "HouseAge"),
    description="Features",
    rows=8,
    layout=widgets.Layout(width="420px"),
)

reg_test_size_widget = widgets.FloatSlider(
    value=0.2,
    min=0.1,
    max=0.5,
    step=0.05,
    description="Test size",
    continuous_update=False,
)


def run_regression(selected_features, test_size):
    if len(selected_features) == 0:
        display(Markdown("❗Please select at least one feature."))
        return

    X = df[list(selected_features)]
    y = df["MedHouseVal"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)

    display(Markdown(f"**Selected features:** `{list(selected_features)}`"))
    display(Markdown(f"**Test size:** `{test_size:.2f}`"))
    display(Markdown(f"**MSE:** `{mse:.4f}` (lower is better)"))

reg_output = widgets.interactive_output(
    run_regression,
    {"selected_features": reg_feature_widget, "test_size": reg_test_size_widget},
)

display(widgets.VBox([reg_feature_widget, reg_test_size_widget, reg_output]))


### Regression reflection questions
Answer in complete sentences:

1. Which feature set gave your **lowest MSE**?
2. Did adding more features always improve performance? Why/why not?
3. Which features appeared least helpful?
4. What does this suggest about feature selection?


# Part 2) Supervised Learning — Classification

Now we predict flower species in the Iris dataset.

### Features
- sepal length (cm)
- sepal width (cm)
- petal length (cm)
- petal width (cm)

### Your task
Use widgets to change selected features and tree depth, then compare accuracy.


In [ ]:
# Load and preview Iris data
iris = load_iris(as_frame=True)
flower_df = iris.frame.copy()
flower_df["species"] = flower_df["target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica",
})

display(flower_df.head())
print(f"Rows: {flower_df.shape[0]}, Columns: {flower_df.shape[1]}")


## Interactive Decision Tree Lab

### Exactly what to do
1. Select 1+ features.
2. Move **Max depth** to control tree complexity.
3. Compare **Accuracy** after each change.

### Interpretation tip
- Higher accuracy is better on this held-out test set.
- Very large depth may overfit in many problems.


In [ ]:
all_classification_features = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]

clf_feature_widget = widgets.SelectMultiple(
    options=all_classification_features,
    value=("sepal length (cm)", "petal length (cm)"),
    description="Features",
    rows=4,
    layout=widgets.Layout(width="420px"),
)

clf_depth_widget = widgets.IntSlider(
    value=2,
    min=1,
    max=10,
    step=1,
    description="Max depth",
    continuous_update=False,
)


def run_classifier(selected_features, max_depth):
    if len(selected_features) == 0:
        display(Markdown("❗Please select at least one feature."))
        return

    X = flower_df[list(selected_features)]
    y = flower_df["species"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    display(Markdown(f"**Selected features:** `{list(selected_features)}`"))
    display(Markdown(f"**Max depth:** `{max_depth}`"))
    display(Markdown(f"**Accuracy:** `{acc:.4f}` (higher is better)"))

clf_output = widgets.interactive_output(
    run_classifier,
    {"selected_features": clf_feature_widget, "max_depth": clf_depth_widget},
)

display(widgets.VBox([clf_feature_widget, clf_depth_widget, clf_output]))


### Classification reflection questions
Answer in complete sentences:

1. Which single feature gave your best accuracy?
2. How did changing `max_depth` affect performance?
3. If we had many more flower types and features, what features might matter most?
4. Would you expect the best depth to be smaller, larger, or data-dependent? Explain.


## Submission checklist
- [ ] I ran all cells in order.
- [ ] I completed at least 4 regression trials.
- [ ] I completed at least 4 classification trials.
- [ ] I answered every reflection question clearly.
